In [ ]:
# 导入依赖库
import pandas as pd
import numpy as np
import scanpy as sc                 # 单细胞/空间转录组核心分析库
import anndata as ad                # 存储单细胞数据的格式（AnnData）
import scipy.io                     # 读取矩阵文件（.mat）
import matplotlib.pyplot as plt
import os                           # 操作电脑文件/文件夹
import sys                          # 系统相关
import torch
import random
import warnings
warnings.filterwarnings("ignore")

# 固定路径配置（适配你的数据位置）
PROJECT_ROOT = r"D:\vscodeprojects\STINR_remake"                            # 项目根目录
DATA_DIR = os.path.join(PROJECT_ROOT, "data")                               # 存放原始空间转录组数据
ANNOT_DIR = os.path.join(DATA_DIR, "DLPFC_annotations")                     # 存放细胞类型注释文件（DLPFC 人脑皮层数据）
os.makedirs(os.path.join(PROJECT_ROOT, "results_DLPFC"), exist_ok=True)     # 创建结果保存文件夹

# 随机种子固定
seed = 1
torch.manual_seed(seed) 
torch.cuda.manual_seed(seed) 
torch.cuda.manual_seed_all(seed)  
np.random.seed(seed)  
random.seed(seed)  
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# 基础参数
slice_idx = [151673, 151674, 151675, 151676]                                 # 4 个空间转录组切片编号（人类大脑皮层 DLPFC 公开数据）                                
celltype_list_use = ['Astros_1', 'Astros_2', 'Astros_3',                     # 星形胶质细胞
                     'Endo', 'Micro/Macro',                                  # 内皮细胞、小胶质/巨噬细胞
                     'Oligos_1', 'Oligos_2', 'Oligos_3',                     # 少突胶质细胞
                     'Ex_1_L5_6', 'Ex_2_L5', 'Ex_3_L4_5',                    # 兴奋性神经元（皮层分层 L2-L6）
                     'Ex_4_L_6', 'Ex_5_L5',
                     'Ex_6_L4_6', 'Ex_7_L4_6', 'Ex_8_L5_6', 
                     'Ex_9_L5_6', 'Ex_10_L2_4']

# 检查路径是否存在
print(f"数据目录是否存在: {os.path.exists(DATA_DIR)}")
print(f"注释文件目录是否存在: {os.path.exists(ANNOT_DIR)}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import math

class DenseLayer(nn.Module):
    def __init__(self, 
                 c_in, 
                 c_out, 
                 zero_init=False, 
                 ):
        super().__init__()
        self.linear = nn.Linear(c_in, c_out)
        # 初始化
        if zero_init:                                                # 用0初始化权重
            nn.init.zeros_(self.linear.weight.data)
        else:                                                        # Xavier 均匀初始化
            nn.init.uniform_(self.linear.weight.data, -np.sqrt(
                6 / (c_in + c_out)), np.sqrt(6 / (c_in + c_out)))
        nn.init.zeros_(self.linear.bias.data)                        # 用0初始化偏置项

    def forward(self, node_feats):
        node_feats = self.linear(node_feats)
        return node_feats

class SineLayer(nn.Module):
    def __init__(self, c_in, c_out, bias=True, zero_init=False, omega_0=1):
        super().__init__()
        self.omega_0 = omega_0
        self.zero_init = zero_init
        self.in_features = c_in
        self.linear = nn.Linear(c_in, c_out, bias=bias)
        
        if self.zero_init:
            nn.init.zeros_(self.linear.weight.data)
        else:
            nn.init.uniform_(self.linear.weight.data, -np.sqrt(
                6 / (c_in + c_out)), np.sqrt(6 / (c_in + c_out)))
            nn.init.zeros_(self.linear.bias.data)
            
    def forward(self, input):
        return torch.sin(self.omega_0 * self.linear(input))

class DeconvNet(nn.Module):
    def __init__(self, 
                 hidden_dims,       # 网络隐藏层维度
                 n_celltypes,       # 细胞类型数量（19种）
                 n_slices,          # 切片个数（4个）
                 n_heads,           # 注意力头
                 slice_emb_dim,     # 切片嵌入维度
                 adj_dim,           # 邻接矩阵维度
                 coef_fe,           # 特征损失的权重
                 ):
        seed=1                              # 固定随机种子
        torch.manual_seed(seed) 
        torch.cuda.manual_seed(seed) 
        torch.cuda.manual_seed_all(seed)  
        np.random.seed(seed)  
        random.seed(seed)  
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

        super().__init__()
        self.training_steps = 14001         # 训练14001步
        mid_channel = 200                   # 中间层通道200维
        
        self.encoder_layer0 = nn.Sequential(SineLayer(3, mid_channel),                  # 输入3维坐标 V（R^N*3）
                                            SineLayer(mid_channel, mid_channel),        # g_w
                                            SineLayer(mid_channel, 30),
                                            DenseLayer(30, hidden_dims[0]))
                    
        self.encoder_layer1 = DenseLayer(hidden_dims[0],hidden_dims[2])                 # Z_latent
        
        self.decoder = nn.Sequential(SineLayer(hidden_dims[2], mid_channel),            # h_φ
                                            DenseLayer(mid_channel, hidden_dims[0]))
        
        self.deconv_alpha_layer = DenseLayer(hidden_dims[2] + slice_emb_dim,            # alpha层 调整spot基因表达量
                                             1, zero_init=True)
        
        self.deconv_beta_layer = nn.Sequential(DenseLayer(hidden_dims[2], n_celltypes,  # beta层 调整cell类型表达量 β=b_ψ​(Z)+[γ,⋯,γ]^T
                                                          zero_init=True))              # P

        self.gamma = nn.Parameter(torch.Tensor(n_slices,                                # gamma层 调整整体
                                               hidden_dims[0]).zero_())

        self.slice_emb = nn.Embedding(n_slices, slice_emb_dim) # n_slice, 16
        self.coef_fe = coef_fe

    def forward(self, 
                coord, # n1*3 coordinate matrix
                adj_matrix,
                node_feats, # input node features n1*n2
                count_matrix, # gene expression counts
                library_size, # library size (based on Y)
                slice_label, # slice label
                basis, # basis matrix
                step
                ):
        # encoder
        self.node_feats = node_feats
        self.coord = coord/100 # 坐标缩放
        
        Z, mid_fea = self.encoder(node_feats)

        # deconvolutioner
        slice_label_emb = self.slice_emb(slice_label)
        beta, alpha = self.deconvolutioner(Z, slice_label_emb)
        self.node_feats_recon = self.decoder(Z)

        # 反卷积损失
        log_lam = torch.log(torch.matmul(beta, basis) + 1e-6) + alpha + self.gamma[slice_label]                 # ∑P·B+β (α_i + γ_s = 论文里的 β_{i,g}(β=b_ψ​(Z)+[γ,⋯,γ]^T ))
        lam = torch.exp(log_lam)                                                                                # λ_{i,g}
        self.decon_loss = - 5*torch.mean(torch.sum(count_matrix *                                               # 泊松负对数似然损失
                                        (torch.log(library_size + 1e-6) + log_lam                               # Ti = library_size
                                         ) - library_size * lam, axis=1))
        
        self.fea_loss = 1*torch.norm(node_feats-mid_fea, 2)+2*torch.norm(node_feats-self.node_feats_recon, 2)   # Lrec & Lreg 
        
        # 总损失
        loss = 1*(self.decon_loss + self.fea_loss)  
        
        denoise = torch.matmul(beta, basis) # Xrec β·B = ∑P·B（对应论文里的求和项）
        return loss, mid_fea, denoise, Z, 0, 0

    def evaluate(self, adj_matrix, coord, node_feats, slice_label):
        slice_label_emb = self.slice_emb(slice_label)
        Z, _ = self.encoder(node_feats)
        beta, alpha = self.deconvolutioner(Z, slice_label_emb)
        return Z, beta, alpha, self.gamma
            
    def encoder(self, H):
        self.mid_fea = self.encoder_layer0(self.coord)
        Z = self.encoder_layer1(self.mid_fea)
        return Z, self.mid_fea
    
    def deconvolutioner(self, Z, slice_label_emb):
        beta = self.deconv_beta_layer(torch.sin(Z))
        beta = F.softmax(beta, dim=1)
        H = torch.sin(torch.cat((Z, slice_label_emb), axis=1))
        alpha = self.deconv_alpha_layer(H)
        return beta, alpha

In [3]:
from tqdm import tqdm
from sklearn.metrics.cluster import adjusted_rand_score
from sklearn.mixture import GaussianMixture

class Model():
    def __init__(self, adata_st_list_raw, adata_st, adata_basis, 
                 hidden_dims=[512, 128],
                 n_heads=1,
                 slice_emb_dim=16,
                 coef_fe=0.1,
                 training_steps=11,
                 lr=0.001,
                 seed=112,
                 distribution="Poisson"
                 ):
        self.seed = seed
        self.adata_basis = adata_basis
        # 固定种子
        torch.manual_seed(seed) 
        torch.cuda.manual_seed(seed) 
        torch.cuda.manual_seed_all(seed)  
        np.random.seed(seed)  
        random.seed(seed)  
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        
        self.lr = lr
        self.training_steps = training_steps
        self.adata_st = adata_st
        self.celltypes = list(adata_basis.obs.index)
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        print(f"使用设备: {self.device}")
        
        self.hidden_dims = [adata_st.shape[1]] + hidden_dims
        self.n_celltype = adata_basis.shape[0]
        self.n_slices = len(sorted(set(adata_st.obs["slice"].values)))
        
        # 初始化网络
        if distribution == "Poisson":
            self.net = DeconvNet(hidden_dims=self.hidden_dims,
                                 n_celltypes=self.n_celltype,
                                 n_slices=self.n_slices,
                                 n_heads=n_heads,
                                 slice_emb_dim=slice_emb_dim,
                                 adj_dim=torch.from_numpy(np.array(adata_st.obsm["graph"])).float().to(self.device).shape[1],
                                 coef_fe=coef_fe,
                                 ).to(self.device)
        else:
            raise NotImplementedError("NB分布暂未实现")
        
        self.optimizer = optim.Adamax(list(self.net.parameters()), lr=lr)
        
        # 数据转换为tensor
        if scipy.sparse.issparse(adata_st.X):
            self.X = torch.from_numpy(adata_st.X.toarray()).float().to(self.device)
        else:
            self.X = torch.from_numpy(adata_st.X).float().to(self.device)
        self.A = torch.from_numpy(np.array(adata_st.obsm["graph"])).float().to(self.device)
        self.Y = torch.from_numpy(np.array(adata_st.obsm["count"])).float().to(self.device)
        self.lY = torch.from_numpy(np.array(adata_st.obs["library_size"].values.reshape(-1, 1))).float().to(self.device)
        self.slice = torch.from_numpy(np.array(adata_st.obs["slice"].values)).long().to(self.device)
        self.basis = torch.from_numpy(np.array(adata_basis.X)).float().to(self.device)
        self.coord = torch.from_numpy(np.array(adata_st.obsm['3D_coor'])).float().to(self.device)
        
    def train(self, report_loss=True, step_interval=500):
        self.net.train()
        
        for step in tqdm(range(self.training_steps), desc="训练进度"):
            loss, recon, denoise, Z_, ind_min, ind_max = self.net(coord=self.coord,
                                          adj_matrix=self.A, 
                                          node_feats=self.X, 
                                          count_matrix=self.Y, 
                                          library_size=self.lY, 
                                          slice_label=self.slice, 
                                          basis=self.basis,
                                          step=step)
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            # 每1000步评估并绘图
            if step % 1000 == 0:
                # 加载原始数据
                adata_st_list_raw0 = ad.read_h5ad(os.path.join(DATA_DIR, 'adata_st_list_raw0.h5ad'))
                adata_st_list_raw1 = ad.read_h5ad(os.path.join(DATA_DIR, 'adata_st_list_raw1.h5ad'))
                adata_st_list_raw2 = ad.read_h5ad(os.path.join(DATA_DIR, 'adata_st_list_raw2.h5ad'))
                adata_st_list_raw3 = ad.read_h5ad(os.path.join(DATA_DIR, 'adata_st_list_raw3.h5ad'))
                adata_st_list_raw = [adata_st_list_raw0, adata_st_list_raw1, adata_st_list_raw2, adata_st_list_raw3]
                
                # 评估
                result = self.eval(adata_st_list_raw, save=False, output_path=os.path.join(PROJECT_ROOT, "results_DLPFC"))
                np.random.seed(self.seed)
                gm = GaussianMixture(n_components=7, covariance_type='tied', reg_covar=1e-4, init_params='kmeans')
                y = gm.fit_predict(self.latent, y=None)
                self.adata_st.obs["GM"] = [str(z) for z in y]
                
                # 计算ARI并绘图
                for i in range(len(result)):
                    result[i].obs["GM"] = self.adata_st.obs.loc[result[i].obs_names, ]["GM"]
                    section_id = str(slice_idx[i])
                    # 加载注释文件
                    Ann_df = pd.read_csv(os.path.join(ANNOT_DIR, f"{section_id}_truth.txt"), sep='\t', header=None, index_col=0)
                    Ann_df.columns = ['Ground Truth']
                    result[i].obs_names = [z[:-7] for z in result[i].obs_names]
                    result[i].obs['Ground Truth'] = Ann_df.loc[result[i].obs_names, 'Ground Truth']
                    obs_df = result[i].obs.dropna()
                    ari = adjusted_rand_score(obs_df["GM"], obs_df['Ground Truth'])
                    print(f"Step {step} - Slice {slice_idx[i]} ARI: {ari:.3f}")
                    
                    # 绘制空间分布图
                    plt.rcParams['figure.figsize'] = (12, 6)
                    sc.pl.spatial(result[i], img_key="hires", color=["GM", "Ground Truth"], 
                                  title=[f'Slice {slice_idx[i]} results ARI={round(ari,3)}', 'Ground Truth'],
                                  show=True)
                
    def eval(self, adata_st_list_raw, save=False, output_path="./results"):
        self.net.eval()
        with torch.no_grad():
            self.Z, self.beta, self.alpha, self.gamma = self.net.evaluate(self.A, self.coord, self.X, self.slice)
        
        # 提取latent特征
        embeddings = self.Z.detach().cpu().numpy()
        cell_reps = pd.DataFrame(embeddings, index=self.adata_st.obs.index)
        self.adata_st.obsm['latent'] = cell_reps.values
        self.latent = cell_reps.values
        
        # 保存结果（如果需要）
        if save:
            os.makedirs(output_path, exist_ok=True)
            cell_reps.to_csv(os.path.join(output_path, "representation.csv"))
        
        # 反卷积结果分配
        b = self.beta.detach().cpu().numpy()
        n_spots = 0
        adata_st_decon_list = []
        for i, adata_st_i in enumerate(adata_st_list_raw):
            adata_st_i.obs.index = adata_st_list_raw[i].obs.index + f"-slice{i}"
            decon_res = pd.DataFrame(b[n_spots:n_spots+len(adata_st_i)], columns=self.celltypes, index=adata_st_i.obs.index)
            adata_st_i.obs = adata_st_i.obs.join(decon_res)
            n_spots += len(adata_st_i)
            adata_st_decon_list.append(adata_st_i)
        return adata_st_decon_list

In [ ]:
# 加载数据
adata_st_list_raw0 = ad.read_h5ad(os.path.join(DATA_DIR, 'adata_st_list_raw0.h5ad'))
adata_st_list_raw1 = ad.read_h5ad(os.path.join(DATA_DIR, 'adata_st_list_raw1.h5ad'))
adata_st_list_raw2 = ad.read_h5ad(os.path.join(DATA_DIR, 'adata_st_list_raw2.h5ad'))
adata_st_list_raw3 = ad.read_h5ad(os.path.join(DATA_DIR, 'adata_st_list_raw3.h5ad'))
adata_st_list_raw = [adata_st_list_raw0, adata_st_list_raw1, adata_st_list_raw2, adata_st_list_raw3]    # 4个组数据

adata_st = ad.read_h5ad(os.path.join(DATA_DIR, 'adata_st_DLPFC.h5ad'))                                  # 整合数据
adata_basis = ad.read_h5ad(os.path.join(DATA_DIR, 'adata_basis_DLPFC.h5ad'))                            # Single_B

# 初始化并训练模型
model = Model(adata_st_list_raw, adata_st, adata_basis)
model.train()

# 结果保存路径
save_path = os.path.join(PROJECT_ROOT, "results_DLPFC")
os.makedirs(save_path, exist_ok=True)

# 模型评估
result = model.eval(adata_st_list_raw, save=True, output_path=save_path)

# 高斯混合聚类
np.random.seed(1234)
gm = GaussianMixture(n_components=7, covariance_type='tied', reg_covar=1e-4, init_params='kmeans')
y = gm.fit_predict(model.adata_st.obsm['latent'], y=None)
model.adata_st.obs["GM"] = y
model.adata_st.obs["GM"].to_csv(os.path.join(save_path, "clustering_result.csv"))

# 聚类结果整理
order = [0,1,2,3,4,5,6]
model.adata_st.obs["Cluster"] = [order[label] for label in model.adata_st.obs["GM"].values]
for i in range(len(result)):
    result[i].obs["GM"] = model.adata_st.obs.loc[result[i].obs_names, ]["GM"]
    result[i].obs["Cluster"] = model.adata_st.obs.loc[result[i].obs_names, ]["Cluster"]

# 绘制最终的细胞类型反卷积空间图
for i, adata_st_i in enumerate(result):
    print("="*50)
    print(f"Slice {slice_idx[i]} cell-type deconvolution result:")
    print(list(adata_basis.obs.index))
    plt.rcParams['figure.figsize'] = (20, 15)
    sc.pl.spatial(adata_st_i, img_key="lowres", color=list(adata_basis.obs.index), size=1., show=True)